# Day 48-49 — PROJECT 2: NLP Text Classifier
### Fine-tune DistilBERT on BBC News → Serve via FastAPI → Dockerize

## Part 1 (Day 48): Load Data & Fine-tune DistilBERT
### 1. Setup & Imports

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# install transformers/datasets if needed
for pkg in ['transformers', 'datasets', 'accelerate']:
    try:
        __import__(pkg)
    except:
        import subprocess
        subprocess.run(['pip', 'install', pkg, '--quiet'])

from transformers import (DistilBertTokenizerFast, DistilBertForSequenceClassification,
                          Trainer, TrainingArguments)
from datasets import Dataset

plt.style.use('dark_background')

print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"Device:       {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("Ready! ✅")

PyTorch:      2.12.0+cpu
CUDA:         False
Device:       cpu
Ready! ✅


## 2. Load Data from Folders → Build DataFrame

In [2]:
import os
import glob
import pandas as pd

data_dir = r'C:\DS-AI-75d\archive\bbc'
categories = ['business', 'entertainment', 'politics', 'sport', 'tech']

texts = []
labels = []

for cat in categories:
    folder = os.path.join(data_dir, cat)
    files = glob.glob(os.path.join(folder, '*.txt'))
    print(cat, len(files))
    for f in files:
        try:
            with open(f, 'r', encoding='utf-8') as file:
                content = file.read()
        except UnicodeDecodeError:
            with open(f, 'r', encoding='latin-1') as file:
                content = file.read()
        texts.append(content.strip())
        labels.append(cat)

df = pd.DataFrame({'text': texts, 'category': labels})
df.shape

business 510
entertainment 386
politics 417
sport 511
tech 401


(2225, 2)

In [3]:
df['category'].value_counts()

category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

In [4]:
df.head()

,text,category
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [5]:
label_map = {cat: i for i, cat in enumerate(categories)}
id2label = {v: k for k, v in label_map.items()}
df['label'] = df['category'].map(label_map)
label_map

{'business': 0, 'entertainment': 1, 'politics': 2, 'sport': 3, 'tech': 4}

## 3. Train/test split + check token lengths

In [6]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label'])

train_df.shape, test_df.shape

((1780, 3), (445, 3))

In [7]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

lengths = [len(tokenizer.encode(t)) for t in df['text'].sample(200, random_state=42)]
max(lengths), sum(lengths)/len(lengths)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (531 > 512). Running this sequence through the model will result in indexing errors


(5303, 517.885)

In [8]:
def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

train_ds = train_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

train_ds

Map:   0%|          | 0/1780 [00:00<?, ? examples/s]

Map:   0%|          | 0/445 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1780
})

In [9]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=5, id2label=id2label, label2id=label_map)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.argmax(axis=1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='./bbc_model_checkpoints',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

In [12]:
import sys
!{sys.executable} -m pip install accelerate --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.108550,0.090854,0.979775
2,0.015845,0.069482,0.986517


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=446, training_loss=0.16160139401397364, metrics={'train_runtime': 2668.7697, 'train_samples_per_second': 1.334, 'train_steps_per_second': 0.167, 'total_flos': 235804584652800.0, 'train_loss': 0.16160139401397364, 'epoch': 2.0})

In [12]:
preds = trainer.predict(test_ds)
y_pred = preds.predictions.argmax(axis=1)
y_true = test_ds['label']

print(classification_report(y_true, y_pred, target_names=categories))

               precision    recall  f1-score   support

     business       0.99      0.95      0.97       102
entertainment       0.99      1.00      0.99        77
     politics       0.97      0.99      0.98        84
        sport       1.00      1.00      1.00       102
         tech       0.99      1.00      0.99        80

     accuracy                           0.99       445
    macro avg       0.99      0.99      0.99       445
 weighted avg       0.99      0.99      0.99       445



In [13]:
cm = confusion_matrix(y_true, y_pred)
cm

array([[ 97,   1,   3,   0,   1],
       [  0,  77,   0,   0,   0],
       [  1,   0,  83,   0,   0],
       [  0,   0,   0, 102,   0],
       [  0,   0,   0,   0,  80]])

In [14]:
trainer.save_model('./bbc_distilbert_final')
tokenizer.save_pretrained('./bbc_distilbert_final')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bbc_distilbert_final\\tokenizer_config.json',
 './bbc_distilbert_final\\tokenizer.json')

In [15]:
import json
with open('./bbc_distilbert_final/label_map.json', 'w') as f:
    json.dump({'label_map': label_map, 'id2label': id2label}, f)

In [16]:
sample_text = "The stock market rallied today after the central bank announced interest rate cuts."

inputs = tokenizer(sample_text, return_tensors='pt', truncation=True, padding='max_length', max_length=256)
with torch.no_grad():
    output = model(**inputs)

pred_id = output.logits.argmax(dim=1).item()
id2label[pred_id]

'business'